# UCS420 – Cognitive Computing
## Assignment 4 | A Cognitive FAQ System Using Pandas (Nova 2.0)

**Student:** Parth Khurana  
**Roll Number:** 1024170354  
**Batch:** 3Q33

---
This assignment builds a small FAQ knowledge base using Pandas, scores user queries against the FAQ entries, filters entries by category, updates a keyword, saves the DataFrame to CSV, and handles tied top scores.

## Q1 — Build the personalized knowledge base

The last two digits of my roll number are **5** and **4**. The categories are selected using `categories[digit % 3]`, where the category list is `['billing', 'account', 'general']`. Therefore, digit 5 maps to `general` and digit 4 maps to `account`.

In [ ]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

roll_number = "1024170354"
categories = ["billing", "account", "general"]
last_two_digits = [int(digit) for digit in roll_number[-2:]]

fixed_entries = [
    {"question": "what is the annual fee", "answer": "The annual fee is Rs 500.",
     "keywords": "fee cost price charge", "category": "billing"},
    {"question": "how to reset password", "answer": "Go to Settings > Reset Password.",
     "keywords": "password reset login", "category": "account"},
    {"question": "what are your working hours", "answer": "We are open 9 AM to 5 PM.",
     "keywords": "hours timing open time", "category": "general"},
    {"question": "how can i pay the fee", "answer": "You can pay via UPI, card, or net banking.",
     "keywords": "pay payment upi fee", "category": "billing"},
]

personalized_entries = []
for digit in last_two_digits:
    category = categories[digit % 3]
    if category == "general":
        entry = {
            "question": "where can i find the college office",
            "answer": "The college office is in the main administrative building.",
            "keywords": "college office location administration",
            "category": category
        }
    elif category == "account":
        entry = {
            "question": "how do i update my registered mobile number",
            "answer": "Open your account settings, choose Profile, and update your registered mobile number.",
            "keywords": "mobile number update profile",
            "category": category
        }
    else:
        entry = {
            "question": "how can i check my payment status",
            "answer": "Open the billing section and check the latest status of your payment.",
            "keywords": "payment status bill transaction",
            "category": category
        }
    personalized_entries.append(entry)

df = pd.DataFrame(fixed_entries + personalized_entries)
print("Final FAQ DataFrame (6 entries):")
display(df)

## Q2 — Generate and score a hypothesis

The scoring function compares the query words with each entry's question and keywords. The confidence score is the fraction of unique query words found in that entry. Entries with a score above zero are returned from highest to lowest score.

In [ ]:
def score_query(query, data):
    query_words = set(query.lower().split())
    results = []

    for _, row in data.iterrows():
        searchable_text = (row["question"] + " " + row["keywords"]).lower()
        matched_words = [word for word in query_words if word in searchable_text]
        score = len(matched_words) / len(query_words) if query_words else 0
        if score > 0:
            results.append({
                "question": row["question"],
                "answer": row["answer"],
                "category": row["category"],
                "matched_words": ", ".join(sorted(matched_words)),
                "confidence": round(score, 2)
            })

    results_df = pd.DataFrame(results)
    if not results_df.empty:
        results_df = results_df.sort_values(
            by=["confidence", "question"], ascending=[False, True]
        ).reset_index(drop=True)
    return results_df

query = "reset password"
print("Query:", query)
display(score_query(query, df))

## Q3 — Return questions from the same category

I use the category of the first personalized entry (`general`) and return all FAQ entries belonging to that category.

In [ ]:
def same_category(category_name, data):
    return data.loc[data["category"].str.lower() == category_name.lower(),
                    ["question", "answer", "keywords", "category"]]

personalized_category = personalized_entries[0]["category"]
print("Category selected:", personalized_category)
display(same_category(personalized_category, df))

## Q4 — Add a keyword and save the updated DataFrame

This cell asks for a new keyword, adds it to the first FAQ entry's keywords, and saves all six rows to `1024170354_faq_data.csv`.

In [ ]:
entry_index = 0
new_keyword = input("Enter a new keyword to add to the first FAQ entry: ").strip()

if new_keyword:
    existing_keywords = df.at[entry_index, "keywords"].split()
    if new_keyword.lower() not in [word.lower() for word in existing_keywords]:
        df.at[entry_index, "keywords"] = df.at[entry_index, "keywords"] + " " + new_keyword
    else:
        print("That keyword is already present.")

csv_filename = f"{roll_number}_faq_data.csv"
df.to_csv(csv_filename, index=False)
print(f"Updated FAQ DataFrame saved to {csv_filename}")
display(df)

## Q5 — Count FAQ entries per category

The `groupby()` operation groups the entries by category and counts the FAQ questions in each group.

In [ ]:
category_counts = df.groupby("category").size().reset_index(name="number_of_entries")
display(category_counts)

## Q6 — Show all entries tied for the highest score

This version prints every entry with the highest positive confidence score when multiple entries tie. It also demonstrates a tie using `fee` and a non-tied query using `reset password`.

In [ ]:
def score_query_with_ties(query, data):
    results = score_query(query, data)

    if results.empty:
        print("No matching FAQ entries found.")
        return results

    highest_score = results["confidence"].max()
    top_matches = results[results["confidence"] == highest_score]

    if len(top_matches) > 1:
        print(f"Tie: {len(top_matches)} entries share the highest confidence score ({highest_score}).")
        display(top_matches)
    else:
        print(f"One best match found (confidence: {highest_score}).")
        display(top_matches)

    return results

print("Demonstration 1 — query that produces a tie:")
tie_results = score_query_with_ties("fee", df)

print("Demonstration 2 — query that does not produce a tie:")
non_tie_results = score_query_with_ties("reset password", df)

## Conclusion

The FAQ system stores exactly six entries in a Pandas DataFrame, including two entries personalized using the last two digits of the roll number. It ranks FAQ matches by confidence, filters entries by category, lets the user add a keyword and saves the updated data to CSV, counts entries per category, and displays all equally ranked top matches when a tie occurs.